# 01 · Setup and data exploration

**Missions covered:** 2 (cache) ✅ · 3 (loading a session) ✅ · 4 (exploring the laps) ✅

**Goal:** set up the environment, load a Formula 1 session with [FastF1](https://docs.fastf1.dev/)
and understand the structure of the data before any analysis.

**Session studied:** 2024 Monaco Grand Prix qualifying.

**Prerequisites**
- The `.venv` virtual environment is activated and the dependencies from `requirements.txt` are installed.
- Jupyter Lab is started from the root of the repo (the notebook itself runs from `notebooks/`).
- Lap and telemetry data require a (free) Formula 1 account.
  Sign in from a terminal: `python -m fastf1 auth f1tv --authenticate`

## Imports

| Library | Role in the project |
|---|---|
| `os` | handle file paths and folders |
| `fastf1` | download official Formula 1 data |
| `fastf1.plotting` | official driver and tyre colours (used from notebook 03) |
| `numpy` | numerical computing (statistics, conversions) |
| `pandas` | work with data tables (*DataFrames*) |
| `matplotlib.pyplot` | draw charts |
| `display` | show a table properly in Jupyter (instead of `print`) |

In [ ]:
import os

import fastf1
import fastf1.plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

## Mission 2 ✅ · Enabling the FastF1 cache

**Why a cache?** Each session weighs several tens of MB. Without a cache, FastF1 would download it again
on every run. With the cache, the first load is slow and the following ones are almost instant.
FastF1 stores files by year, Grand Prix and session (`cache/2024/2024-05-26_Monaco_Grand_Prix/...`).

**What the cell does, line by line:**
1. `assert ...`: the notebook runs from `notebooks/`, so `../cache` points to the `cache/` folder at the root of the repo.
   This is checked **before** creating anything, so a `cache/` folder is never created in the wrong place.
   If the condition is false, execution stops with a clear message.
2. `os.path.abspath(...)`: turns the relative path into an absolute path, once, in a single variable.
3. `os.makedirs(..., exist_ok=True)`: creates the folder if it does not exist, without an error if it already does.
   This is required, because FastF1 refuses a cache folder that does not exist.
4. `print(...)`: lets you check visually that the path ends with `f1-telemetry-analysis/cache`.
5. `fastf1.Cache.configure(cache_dir=...)`: enables the cache. Since FastF1 3.9, this function replaces `enable_cache`
   (deprecated). The parameter must be passed **by name** (`cache_dir=`), because it comes after a `*` in the function definition.

Thanks to the relative path, the project works wherever the repo is cloned (Linux, macOS, another folder).
The `cache/` folder is listed in `.gitignore`: it never goes to GitHub.

In [ ]:
# The relative path "../cache" is only correct if the notebook runs from notebooks/
assert os.path.basename(os.getcwd()) == "notebooks", f"Unexpected working directory: {os.getcwd()}"

cache_dir = os.path.abspath("../cache")
os.makedirs(cache_dir, exist_ok=True)
print("FastF1 cache:", cache_dir)

fastf1.Cache.configure(cache_dir=cache_dir)

## Helper function · `show`

`print(df)` turns a table into plain text: columns are cut with `...` and line breaks
make the whole thing unreadable. This function displays a real Jupyter table, with two options:

```python
show(df)                      # the whole table
show(df, n=5)                 # the first 5 rows
show(df, ["col_a", "col_b"])  # only these columns
```

`pd.option_context("display.max_columns", None)` tells pandas not to hide any column,
only for the duration of this display (thanks to the `with`).

> The function is meant for **DataFrames**. To display a simple result (`.shape`, `.columns`…),
> just write it alone on the last line of a cell: Jupyter displays it automatically.

In [ ]:
def show(df, columns=None, n=None):
    '''Display a DataFrame as a Jupyter table, without truncated columns.

    columns : list of column names to keep (all by default)
    n       : number of rows to display (all by default)
    '''
    if columns is not None:
        df = df[columns]
    if n is not None:
        df = df.head(n)
    with pd.option_context("display.max_columns", None):
        display(df)

## Mission 3 ✅ · Loading a session

`fastf1.get_session(year, grand_prix, session_type)` expects three pieces of information:
- **the year**: `2024`;
- **the Grand Prix**: a name (`"Monaco"`) or a round number (`8`);
- **the session type**: `"FP1"`, `"FP2"`, `"FP3"`, `"Q"` (qualifying), `"S"` (sprint), `"R"` (race).

`get_session` does not download anything: it only prepares a `Session` object. `.load()` is what downloads
the data (or reads it from the cache). The two steps are separate so you can choose what to load:
for example `session.load(telemetry=False)` if telemetry is not needed.

The parameters are stored in upper-case **constants** at the top of the cell: to analyse another
race, you only need to change these three lines.

In [ ]:
YEAR = 2024
GRAND_PRIX = "Monaco"
SESSION_TYPE = "Q"

session = fastf1.get_session(YEAR, GRAND_PRIX, SESSION_TYPE)
session.load()
print(session.event["EventName"], "-", session.name)

### Checking that the laps are loaded

Without a Formula 1 account sign-in, FastF1 retrieves the results (from another source, Ergast)
but **not** the laps or the telemetry: accessing `session.laps` then raises a `DataNotLoadedError`.
This cell catches the error with `try` / `except` and prints a clear message instead.

In [ ]:
# Without a Formula 1 account sign-in, FastF1 only retrieves the results:
# accessing session.laps then raises a DataNotLoadedError.
try:
    laps_count = len(session.laps)
except fastf1.exceptions.DataNotLoadedError:
    laps_count = 0

if laps_count == 0:
    print("⚠️ No laps loaded. Sign in from a terminal with:")
    print("   python -m fastf1 auth f1tv --authenticate")
else:
    print(f"✅ {laps_count} laps loaded")

### Qualifying classification

`session.results` has one row per driver and 22 columns (identity, team, times…).
For a classification, only the essentials are kept: position, driver, team and the time of each phase.

A **list** of columns is passed to `show`:
- `df["col"]` returns **a single column** (a `Series`);
- `df[["col_a", "col_b"]]` returns **a table** (a `DataFrame`).

`NaT` (*Not a Time*, the equivalent of `NaN` for durations) means the driver did not take part in that phase:
the last 5 are knocked out in Q1, the next 5 in Q2.

In [ ]:
RANKING_COLUMNS = ["Position", "Abbreviation", "FullName", "TeamName", "Q1", "Q2", "Q3"]
show(session.results, RANKING_COLUMNS)

> ⚠️ **Data issue spotted:** the `Position` column contains 14 twice and 15 twice, and stops at 18
> for 20 drivers. The row order is not reliable at the bottom of the table either: MAG has a Q2 time,
> but appears after ALO and SAR, who were knocked out in Q1. The top of the classification (Q3) is consistent, but this
> column should not be trusted blindly for places 14 to 20.

## Mission 4 ✅ · Exploring `session.laps`

**Key concept: one row = one lap by one driver.**
Each column describes that lap: its time, the tyre used, its number, pit stops…

Before any analysis, we **look** at the data. Otherwise, we risk computing averages on
missing or abnormal values without noticing.

Documentation: [Timing and Telemetry Data → Laps](https://docs.fastf1.dev/core.html#fastf1.core.Laps).

### 4.1 · Size

In [ ]:
session.laps.shape

**426 rows** (one per lap) and **31 columns**. With 20 drivers, that is about 21 laps each on average,
but the actual number varies a lot (see 4.6): qualifying has three phases, and a driver knocked out
in Q1 drives much less than a driver who makes it to Q3.

### 4.2 · Columns

In [ ]:
session.laps.columns

The columns fall into a few families:

| Family | Columns | Content |
|---|---|---|
| Identity | `Driver`, `DriverNumber`, `Team` | who drove the lap |
| Times | `LapTime`, `Sector1Time`, `Sector2Time`, `Sector3Time` | duration of the lap and of its three sectors |
| Timing | `Time`, `LapStartTime`, `LapStartDate`, `Sector*SessionTime` | when the lap happened during the session |
| Pit stops | `PitOutTime`, `PitInTime`, `Stint` | pit exits and entries, stint number |
| Tyres | `Compound`, `TyreLife`, `FreshTyre` | tyre compound, age in laps, new tyre or not |
| Speeds | `SpeedI1`, `SpeedI2`, `SpeedFL`, `SpeedST` | speeds at the measurement points (km/h) |
| Quality | `IsAccurate`, `Deleted`, `DeletedReason`, `FastF1Generated`, `IsPersonalBest` | reliability and validity of the lap |
| Track | `TrackStatus` | flags during the lap (`1` = green, `2` = yellow…) |

### 4.3 · First rows

In [ ]:
show(session.laps, n=10)

The first 10 rows are all Leclerc's laps. They show an **alternating pattern**:
- lap 1 (1:47): **out-lap** from the pits (`PitOutTime` filled in), the tyres are warmed up;
- laps 2 and 4 (about 1:12): **flying laps**, flat out;
- lap 3 (1:37): **cool-down lap**, driven slowly to prepare the next attempt;
- lap 5 (1:56): **in-lap** to the pits (`PitInTime` filled in).

So in qualifying, most laps are **not** fast laps: this is essential to know before computing an average.

### 4.4 · Data types

In [ ]:
session.laps.dtypes

| pandas type | Meaning | Examples |
|---|---|---|
| `object` | text | `Driver`, `Compound`, `Team`, `TrackStatus` |
| `float64` | decimal number | `LapNumber`, `TyreLife`, `SpeedST` |
| `bool` | true / false | `IsAccurate`, `Deleted`, `FreshTyre` |
| `timedelta64` | **duration** | `LapTime`, `Sector1Time`, `PitInTime` |
| `datetime64` | real **date and time** | `LapStartDate` |

**Why isn't `LapTime` a plain number?** Because it is a duration (minutes, seconds, milliseconds).
The `timedelta64` type adds and compares durations correctly (`1:12.4 + 0:19.3`).
For statistics, it will be converted to seconds with `.dt.total_seconds()` (notebook 02).

**Why is `LapNumber` a `float` and not an integer?** numpy's classic integer type cannot
represent a missing value (`NaN`). pandas therefore uses `float` to be safe.

### 4.5 · Missing values

Only the columns with at least one missing value are shown.

In [ ]:
missing = session.laps.isna().sum()
missing[missing > 0]

Every gap has an explanation:
- **`PitOutTime` and `PitInTime` (334 missing out of 426)**: this is **normal**. These columns only have a value for
  out-laps (92) and in-laps (92). Other laps, by definition, have neither.
- **`LapTime` (79 missing)**: 78 of these are **in-laps**. The driver enters the pit lane before
  the finish line: the lap is never completed, so it has no time. The last case is an isolated anomaly
  (SAI, lap 21).
- **`Sector3Time` (78)** and **`SpeedFL` (92)**: same reason. The third sector and the speed at the finish line
  are not measured when the driver goes into the pits.
- **`Position`**: empty on every lap, because this column only makes sense in a race.

### 4.6 · Key columns for what comes next

In [ ]:
session.laps["Driver"].value_counts()

From 30 laps (GAS) down to 11 (ALO, BOT, ZHO). Drivers knocked out in Q1 (ALO, SAR, PER, BOT, ZHO) have 11 or 12 laps;
those who reach Q3 have between 24 and 30.

In [ ]:
session.laps["Compound"].value_counts()

All 426 laps are on **soft tyres** (`SOFT`): in qualifying, everyone looks for maximum grip.

In [ ]:
session.laps["IsAccurate"].value_counts()

185 laps out of 426 (43 %) are flagged as **not accurate**. 184 of them are out-laps or in-laps.

In [ ]:
session.laps["Deleted"].value_counts()

In [ ]:
show(session.laps[session.laps["Deleted"]], ["Driver", "LapNumber", "LapTime", "DeletedReason"], n=5)

21 laps were **deleted** by race control, mostly for exceeding track limits
(`TRACK LIMITS AT TURN 10`, `TURN 1`).

In [ ]:
session.laps["TrackStatus"].value_counts()

`1` means green flag. `12` means the status changed from green (`1`) to **yellow flag** (`2`) during the lap:
47 laps were affected.

In [ ]:
session.laps["TyreLife"].describe()

Tyres are between 1 and 12 laps old: in qualifying, every attempt is made on almost new tyres.

## Exploration summary

- `session.laps` contains **one row per lap**: 426 laps and 31 columns for the 2024 Monaco qualifying.
- Times are **durations** (`timedelta64`): they must be converted to seconds to compute statistics.
- **Most laps are not fast laps**: out-laps, cool-down laps and in-laps.
- **Missing values are not errors**: almost all of them are in-laps.
- Before any comparison, the data must be **filtered** to remove:
  - laps with a pit stop (`PitInTime`, `PitOutTime`);
  - inaccurate laps (`IsAccurate`);
  - deleted laps (`Deleted`);
  - laps under yellow flag (`TrackStatus`).
- In `session.results`, the `Position` column is **inconsistent** for places 14 to 20.